# AI-Orb – deutsche, dynamische GPU-Stimme auf Google Colab

Das originale `sesame/csm-1b` ist ein englisches Modell. Dieses Notebook verwendet deshalb `Chatterbox Multilingual V3`: ein deutschfähiges, ausdrucksstarkes Sprachmodell mit regelbarer Prosodie und optionaler Stimmreferenz.

1. Wähle **Laufzeit → Laufzeittyp ändern → GPU**.
2. Führe die Zellen von oben nach unten aus.
3. Lass die letzte Zelle aktiv, solange der Orb diese Stimme verwenden soll.

> Eine Referenzaufnahme darf nur deine eigene Stimme oder eine ausdrücklich freigegebene Stimme enthalten.


In [ ]:
# 1/5 – GPU prüfen und stabile Abhängigkeiten installieren
import subprocess
import sys

subprocess.run(["apt-get", "-qq", "update"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1"], check=True, stdout=subprocess.DEVNULL)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "git+https://github.com/resemble-ai/chatterbox.git@65b18437192794391a0308a8f705b1e33e633948",
    "fastapi>=0.115", "uvicorn>=0.30",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--force-reinstall", "--no-deps", "torchvision==0.21.0",
])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Keine CUDA-GPU gefunden. Aktiviere in Colab eine GPU-Laufzeit.")
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✓ GPU: {gpu_name} ({gpu_gib:.1f} GiB)")
print("Falls Colab einen Neustart verlangt: Laufzeit neu starten und ab Zelle 2 fortfahren.")


In [ ]:
# 2/5 – Deutschfähiges Ausdrucksmodell laden
import gc
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

MODEL_ID = "ResembleAI/chatterbox-multilingual-v3"
DEVICE = "cuda"
LANGUAGE_ID = "de"
gc.collect()
torch.cuda.empty_cache()
print("Lade Chatterbox Multilingual V3 …")
model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE, t3_model="v3")
SAMPLE_RATE = int(model.sr)
used_gib = torch.cuda.memory_allocated() / 1024**3
print(f"✓ Deutsche Ausdrucksstimme bereit auf {gpu_name}. GPU-Speicher: {used_gib:.2f} GiB")


## Optionale deutsche Stimmreferenz

Für eine feste Stimme kannst du 5–12 Sekunden saubere deutsche Sprache hochladen. Ohne Referenz verwendet das Modell seine natürliche Basisstimme.


In [ ]:
# 3/5 – Optional: erlaubte deutsche Referenzaufnahme einlesen
REFERENZ_VERWENDEN = False  # @param {type:"boolean"}

from pathlib import Path
import subprocess

REFERENCE_PATH = None
if REFERENZ_VERWENDEN:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Bitte genau eine eigene oder freigegebene Audiodatei hochladen.")
    source_path = Path(next(iter(uploaded)))
    converted_path = Path("/content/orb_reference_de.wav")
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(source_path),
        "-t", "15", "-ac", "1", "-ar", "24000", str(converted_path),
    ], check=True)
    REFERENCE_PATH = str(converted_path)
    print(f"✓ Deutsche Stimmreferenz geladen: {REFERENCE_PATH}")
else:
    print("ℹ Keine Referenz: natürliche deutsche Basisstimme wird verwendet.")


In [ ]:
# 4/5 – Dynamische Synthese definieren und auf Deutsch testen
import io
import threading
import time
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

generation_lock = threading.Lock()
EMOTION_EXAGGERATION = {
    "neutral": 0.55, "friendly": 0.58, "curious": 0.62,
    "thinking": 0.48, "happy": 0.72, "excited": 0.78,
    "calm": 0.42, "warning": 0.68, "error": 0.64,
}

def synthesize(text: str, speaker: int = 0, speed: float = 1.0, emotion: str = "neutral") -> tuple[bytes, float]:
    del speaker
    text = " ".join(text.split())
    if not text:
        raise ValueError("Der Text ist leer.")
    if len(text) > 800:
        raise ValueError("Maximal 800 Zeichen pro Anfrage.")
    exaggeration = EMOTION_EXAGGERATION.get(emotion.lower(), 0.55)
    cfg_weight = 0.30 if speed < 0.96 else 0.36
    kwargs = {
        "language_id": LANGUAGE_ID, "exaggeration": exaggeration,
        "temperature": 0.72, "cfg_weight": cfg_weight,
    }
    if REFERENCE_PATH:
        kwargs["audio_prompt_path"] = REFERENCE_PATH
    started = time.perf_counter()
    with generation_lock, torch.inference_mode():
        generated = model.generate(text, **kwargs)
    elapsed = time.perf_counter() - started
    waveform = generated.detach().float().cpu().numpy().squeeze()
    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise RuntimeError("Das Modell hat keine gültigen Audiodaten erzeugt.")
    peak = float(np.max(np.abs(waveform)))
    if peak > 0.98:
        waveform = waveform * (0.98 / peak)
    buffer = io.BytesIO()
    sf.write(buffer, waveform, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    return buffer.getvalue(), elapsed

TESTTEXT = "Hallo. Ich bin der AI-Orb. Hmm … schauen wir uns das gemeinsam an."  # @param {type:"string"}
test_wav, test_seconds = synthesize(TESTTEXT, emotion="friendly")
Path("/content/ai_orb_deutsch_test.wav").write_bytes(test_wav)
print(f"✓ Deutscher Test erzeugt in {test_seconds:.2f} Sekunden.")
display(Audio(test_wav, autoplay=False))


In [ ]:
# 5/5 – Geschützten deutschen TTS-Server und Cloudflare-Tunnel starten
import re
import secrets
import subprocess
import threading
import time
import urllib.request
from pathlib import Path

import uvicorn
from fastapi import FastAPI, Header, HTTPException, Response
from pydantic import BaseModel, Field

PORT = 8800
try:
    from google.colab import userdata
    configured_token = userdata.get("REMOTE_TTS_TOKEN") or ""
except Exception:
    configured_token = ""
API_TOKEN = configured_token.strip() or secrets.token_urlsafe(32)

class TtsRequest(BaseModel):
    text: str = Field(min_length=1, max_length=800)
    speaker: int = Field(default=0, ge=0, le=31)
    speed: float = Field(default=1.0, ge=0.5, le=2.0)
    emotion: str = Field(default="neutral", max_length=32)

app = FastAPI(title="AI-Orb Deutsche Ausdrucksstimme", docs_url=None, redoc_url=None)

@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID, "language": LANGUAGE_ID, "device": gpu_name}

@app.post("/tts")
def tts(request: TtsRequest, authorization: str = Header(default="")):
    if not secrets.compare_digest(authorization, f"Bearer {API_TOKEN}"):
        raise HTTPException(status_code=401, detail="Falscher oder fehlender Token.")
    try:
        wav, elapsed = synthesize(request.text, request.speaker, request.speed, request.emotion)
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error)) from error
    except Exception as error:
        raise HTTPException(status_code=500, detail=f"Spracherzeugung fehlgeschlagen: {error}") from error
    return Response(content=wav, media_type="audio/wav", headers={"X-TTS-Seconds": f"{elapsed:.3f}"})

if "tts_server" in globals():
    tts_server.should_exit = True
    time.sleep(1)
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()

tts_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=tts_server.run, daemon=True).start()
deadline = time.time() + 30
while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        if time.time() > deadline:
            raise RuntimeError("Der lokale Sprachserver ist nicht rechtzeitig gestartet.")
        time.sleep(0.5)

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared
    )
    cloudflared.chmod(0o755)
tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
PUBLIC_URL = ""
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel_process.stdout.readline() if tunnel_process.stdout else ""
    match = url_pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError("Cloudflare-Tunnel wurde unerwartet beendet.")
if not PUBLIC_URL:
    raise RuntimeError("Cloudflare hat keine Tunnel-Adresse geliefert.")

print("\n" + "=" * 72)
print("✓ DEUTSCHE AUSDRUCKSSTIMME IST BEREIT")
print("=" * 72)
print(f"Endpoint: {PUBLIC_URL}")
print(f"Token:    {API_TOKEN}")
print("Diese Zelle offen lassen.\n")
try:
    while tunnel_process.poll() is None:
        time.sleep(300)
        print("✓ Deutscher Sprachserver aktiv", time.strftime("%H:%M:%S"), flush=True)
finally:
    tts_server.should_exit = True
    if tunnel_process.poll() is None:
        tunnel_process.terminate()
